In [4]:
import torch
import os

def verify_checkpoint(filepath):
    print(f"Проверка чекпоинта: {filepath}")
    
    if not os.path.exists(filepath):
        print("ОШИБКА: Файл не найден!")
        return False
        
    try:
        checkpoint = torch.load(filepath, map_location=torch.device('cpu'))
        
        if not isinstance(checkpoint, dict):
            print(f"❌ ОШИБКА: Файл загрузился, но внутри не словарь (dict), а {type(checkpoint)}.")
            return False
            
        expected_keys =[
            "epoch", 
            "model_state_dict", 
            "optimizer_state_dict", 
            "scheduler_state_dict", 
            "best_val_loss"
        ]
        
        missing_keys = [key for key in expected_keys if key not in checkpoint]
        
        if missing_keys:
            print(f"ВНИМАНИЕ: Файл читается, но в нем нет ожидаемых ключей: {missing_keys}")
        else:
            print("УСПЕХ: Чекпоинт цел и имеет правильную структуру!")
            print(f"   - Эпоха сохранения: {checkpoint['epoch']}")
            print(f"   - Значение best_val_loss: {checkpoint['best_val_loss']:.5f}")
            
        return True
        
    except RuntimeError as e:
        print(f"ОШИБКА RuntimeError (возможно, файл поврежден или не до конца записан): {e}")
        return False
    except EOFError as e:
        print(f"ОШИБКА EOFError (файл точно обрезан/недописан): {e}")
        return False
    except Exception as e:
        print(f"НЕИЗВЕСТНАЯ ОШИБКА при загрузке: {e}")
        return False


In [5]:

   # Укажите пути к вашим чекпоинтам
checkpoints_to_test =[
    "../checkpoints/v1/best_model.pth",
    "../checkpoints/v1/last_model.pth",
    "../checkpoints/v2/best_model.pth",
    "../checkpoints/v2/last_model.pth",
]
    
for cp_path in checkpoints_to_test:
    verify_checkpoint(cp_path)
    print("-" * 40)

Проверка чекпоинта: ../checkpoints/v1/best_model.pth
УСПЕХ: Чекпоинт цел и имеет правильную структуру!
   - Эпоха сохранения: 55
   - Значение best_val_loss: 0.70157
----------------------------------------
Проверка чекпоинта: ../checkpoints/v1/last_model.pth
УСПЕХ: Чекпоинт цел и имеет правильную структуру!
   - Эпоха сохранения: 111
   - Значение best_val_loss: 0.70157
----------------------------------------
Проверка чекпоинта: ../checkpoints/v2/best_model.pth
УСПЕХ: Чекпоинт цел и имеет правильную структуру!
   - Эпоха сохранения: 138
   - Значение best_val_loss: 0.66372
----------------------------------------
Проверка чекпоинта: ../checkpoints/v2/last_model.pth
УСПЕХ: Чекпоинт цел и имеет правильную структуру!
   - Эпоха сохранения: 193
   - Значение best_val_loss: 0.66372
----------------------------------------


In [6]:
from plot2eq.core.tokenizer import Tokenizer
from plot2eq.config import TrainConfig
from plot2eq.models.core_model import Plot2EqModel

config = TrainConfig()
tokenizer = Tokenizer()

model = Plot2EqModel(
        vocab_size=len(tokenizer.tokens),
        pad_idx=tokenizer.token_map["<pad>"],
        d_model=config.d_model,
        nhead=config.nhead,
        num_enc_layers=config.num_enc_layers,
        num_dec_layers=config.num_dec_layers,
        max_seq_len=config.max_seq_len,
        dropout=config.dropout,
    )

try:
    checkpoint = torch.load("../checkpoints/v2/best_model.pth", map_location='cpu')
    model.load_state_dict(checkpoint['model_state_dict'])
    print("✅ Веса успешно загружены в модель!")
except Exception as e:
    print(f"❌ Ошибка загрузки весов в модель: {e}")

✅ Веса успешно загружены в модель!
